In [ ]:
import itertools
import json
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.datasets import dataset_ref
from uavloc.filtering.empty_view import EmptyViewModel
from uavloc.plotting.flight import plot_end_error_cdf
from uavloc.simulation.arms import FilterParams, HeadSpec, build_correlator, estimator_factory
from uavloc.simulation.sequence import (all_fixes, flight_metrics, load_flight_set, per_flight_table, run_flights,
                                        village_bootstrap)
from uavloc.track import experiment_dir, load_runs, log_run, set_seed
from sklearn.metrics import roc_auc_score

from uavloc.data.rasters import load_raster
from uavloc.filtering.belief import MIN_WEIGHTED_COVERAGE
from uavloc.filtering.empty_view import coverage_surface, fit_empty_view
from uavloc.matching.correlator import PositionCorrelator
from uavloc.matching.window import SearchWindow, cells_of, score_window
from uavloc.simulation.camera import CameraConfig, RasterCamera
from uavloc.simulation.ins import InsSimulator
from uavloc.simulation.route import sample_route
from uavloc.simulation.sequence import fix_frames

# Params

In [ ]:
EXP = "032-empty-view-likelihood"
SEED = 42
DEVICE = "cpu"                    # CPU-only process: MPS corrupts CPU conv2d in the same process (bug-121)
ROUTE_SET = "sim-flights"
INITIAL_ERROR_M = {"normal": 0.0, "cold": 40.0}
SIGMA0_M = {"normal": 1.0, "cold": 25.0}
CORRELATOR = {
    "method": "segmentation", "seg_architecture": "lraspp",
    "seg_checkpoint_drone": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_checkpoint_sat": "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt",
    "seg_channels": "all", "drone_seg_gsd_m": 0.385, "match_gsd_m": 0.385, "matching": "weighted-ncc",
    "class_weights": [0.0, 1.0, 1.0, 0.5, 0.0, 0.5], "window": "rect", "search_scale": 1.7, "fov_w_deg": 60.0,
    "sat_feature_crop_pct": 0.03, "drone_feature_crop_pct": 0.0,
}
N_PARTICLES = 2000
TEMPERATURE = 0.5                 # 027.12 / 027.4, tuned in 027
Q_POS_M_PER_SQRT_S = 2.0
WINDOW_CAP_HALF_M = 100.0
WINDOW_MARGIN_M = 2.0
HEAD_RUN = "023.3"
HEAD_EXP = "023-learned-correlation-head"
HEAD_ENCODER_CHANNELS = [32, 32, 16]
HEAD_REFINE_CHANNELS = 16
HEAD_CLEAN_VILLAGES = ["bushcha-50p2422-25p8305", "ivannie-50p4543-25p7398", "maidan-50p2512-26p0155",
                       "maksymivka-48p8871-23p9317", "oblaznytsia-49p2703-24p1288", "pidhaichyky-48p5627-25p1788",
                       "pidhorodnie-49p5387-25p5199", "sadzhavka-48p5678-24p7897", "strilche-48p6821-25p5613",
                       "vesnianka-49p7175-27p3183", "zlynets-50p3697-25p7999"]   # villages the 023 head never trained on
MODEL_PATH = "artifacts/empty_view.json"
TUNING_DIR = "artifacts/tuning"
SPLIT = "tuning"
MIN_AUC = 0.75
SHAPE_FRAMES_PER_VILLAGE = 3
RELIABILITY_BINS = 5
TRUTH_OFFSET_RD_M = np.zeros((1, 2))

In [ ]:
set_seed(SEED)
checks = {}

# Data

In [ ]:
set_config, specs = load_flight_set(ROUTE_SET, split=SPLIT)
len(specs)

# Model

In [ ]:
def head_spec() -> HeadSpec:
    return HeadSpec(experiment_dir(HEAD_EXP) / "artifacts" / f"head-{HEAD_RUN}.pt", tuple(HEAD_ENCODER_CHANNELS),
                    HEAD_REFINE_CHANNELS)

In [ ]:
def at_truth(surface) -> float:
    rows, cols, inside = cells_of(surface, TRUTH_OFFSET_RD_M)
    return float(surface.heatmap[rows[0], cols[0]]) if inside[0] else float("nan")

In [ ]:
def shapes_match(head, ncc, frame, window, grid) -> bool:
    drone = head.drone_map(frame)
    shapes = {score_window(c, drone, c.footprint_m(frame), window).heatmap.shape for c in (head, ncc)}
    head.clear_cache()
    return shapes == {grid.heatmap.shape}

In [ ]:
def calibration_rows(spec, head, ncc, camera) -> list[dict]:
    poses = sample_route(spec.waypoints, set_config.speed_mps, set_config.step_s, set_config.altitude_m)
    rows = []
    for k, (pose, frame) in enumerate(fix_frames(poses, camera, InsSimulator(set_config.ins, spec.seed),
                                                 set_config.correction_interval_s)):
        window = SearchWindow(pose.position, frame.yaw_deg, head.search_half_range_m(frame))
        try:
            grid = coverage_surface(head, frame, window)
        except ValueError:   # the patch leaves the map raster
            continue
        rows.append({"village": spec.village, "route_index": spec.route_index, "t_s": pose.t_s,
                     "drone_coverage": grid.coverage, "blind": grid.coverage < MIN_WEIGHTED_COVERAGE,
                     "map_coverage_truth": at_truth(grid), "map_coverage_window_mean": float(grid.heatmap.mean()),
                     "shapes_match": shapes_match(head, ncc, frame, window, grid) if k < SHAPE_FRAMES_PER_VILLAGE else None})
    return rows

# Run

In [ ]:
rows, example = [], None
for village in sorted({s.village for s in specs}):
    head = build_correlator(load_raster({"kind": "multidate", "place": village, "date": set_config.map_date}),
                            CORRELATOR, DEVICE, head_spec())
    ncc = PositionCorrelator(head.features, replace(head.config, matching="weighted-ncc"))
    camera = RasterCamera(load_raster({"kind": "multidate", "place": village, "date": set_config.drone_date}),
                          CameraConfig(set_config.fov_w_deg))
    for spec in (s for s in specs if s.village == village):
        rows += calibration_rows(spec, head, ncc, camera)
frames = pd.DataFrame(rows)
frames.groupby("blind")[["map_coverage_truth", "map_coverage_window_mean"]].describe().T

In [ ]:
known = frames.dropna(subset=["map_coverage_truth"])
model = fit_empty_view(known["map_coverage_truth"].to_numpy(), known["blind"].to_numpy())
auc = float(roc_auc_score(known["blind"], -known["map_coverage_truth"]))
shape_rows = frames["shapes_match"].dropna()
checks["a_grid_shapes"] = {"frames": int(len(shape_rows)), "pass": bool(len(shape_rows) and shape_rows.all())}
checks["b_auc"] = {"auc": auc, "frames": int(len(known)), "blind_share": float(known["blind"].mean()),
                   "pass": bool(auc >= MIN_AUC)}
model, checks

In [ ]:
known = known.assign(predicted=np.exp(model.log_prob_blind(known["map_coverage_truth"].to_numpy())))
known["bin"] = pd.qcut(known["predicted"], RELIABILITY_BINS, duplicates="drop")
reliability = known.groupby("bin", observed=True).agg(frames=("blind", "size"), predicted=("predicted", "mean"),
                                                      observed=("blind", "mean"))
reliability

# Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
grid_c = np.linspace(0, 1, 200)
axes[0].plot(grid_c, np.exp(model.log_prob_blind(grid_c)), label="fit, floored")
axes[0].scatter(known.groupby(pd.cut(known["map_coverage_truth"], 20), observed=True)["map_coverage_truth"].mean(),
                known.groupby(pd.cut(known["map_coverage_truth"], 20), observed=True)["blind"].mean(), color="k", s=12,
                label="observed share blind")
axes[0].set(xlabel="map weighted coverage under the true footprint", ylabel="P(blind)", title=f"AUC {auc:.3f}")
axes[0].legend()
for blind, part in known.groupby("blind"):
    axes[1].hist(part["map_coverage_truth"], bins=40, range=(0, 1), histtype="step", density=True,
                 label="blind frames" if blind else "frames with structure")
axes[1].set(xlabel="map weighted coverage under the true footprint", ylabel="density")
axes[1].legend()
fig.tight_layout()

In [ ]:
Path("artifacts").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)
frames.to_csv("artifacts/calibration-frames.csv", index=False)
Path(MODEL_PATH).write_text(json.dumps({"model": asdict(model), "fit_on": f"{ROUTE_SET} {SPLIT} split",
                                        "frames": int(len(known))}, indent=1))
Path("artifacts/checks.json").write_text(json.dumps(checks, indent=1, default=float))
fig.savefig("figures/empty-view-calibration.png", dpi=110)
pd.Series({name: value["pass"] for name, value in checks.items()})